By default this notebook plots the paper's success-threshold curves from `analysis/paper_data/success_thresholds_curves.json`. To plot your own planning evaluations, arrange their `0_result_planning.json` files as `<results>/<environment>/<method>/0_result_planning.json` (method names as in `conf/analysis/success_thresholds.json`) and run from the repository root:

```bash
pixi run python -m analysis.success_thresholds --results <results> --output logs/thresholds
```

Then set `curves_file` below to the `curves.json` in that output directory. The paper layout requires all nine methods for both Reacher and FetchSlide. Run plotting on CPU from the repository root or `analysis/`. Times New Roman is used when installed; otherwise the default serif font is used. Exports use a fresh `logs/paper_figures/` directory.

In [ ]:
%matplotlib inline
import os
import sys
import json
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D

root = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from analysis.artifacts import create_output

for font in (Path.home() / '.local/share/fonts/msttcorefonts').glob('*.TTF'):
    font_manager.fontManager.addfont(str(font))
try:
    font_manager.findfont('Times New Roman', fallback_to_default=False)
    font_family = 'Times New Roman'
except ValueError:
    font_family = 'serif'
    print('Times New Roman not found; using the default serif font, so text layout may differ from the paper.')
plt.rcdefaults()
plt.rcParams.update({'font.family': font_family, 'font.size': 15,
                     'axes.titlesize': 15, 'pdf.fonttype': 42, 'ps.fonttype': 42})

curves_file = Path('analysis/paper_data/success_thresholds_curves.json')
styles = {'Vanilla': ['DINO-WM', '#AAD2C8', 'o'], 'Temporal Straightening': ['TS-WM', '#FFE650', 'X'], 'Long History': ['DINO-WM + Long History', '#F78C6B', 's'], 'Context Prefix': ['DINO-WM + Context Prefix', '#06D6A0', 'v'], 'Privileged Physics': ['DINO-WM + Privileged Physics', '#42A5F5', 'D'], 'Context': ['DINO-WM + PhysiCal (Ours)', '#FF647B', '^'], 'LeWM': ['LeWM', '#DDBDFC', 'P'], 'LeWM + Context': ['LeWM + PhysiCal (Ours)', '#06D6A0', '*'], 'Estimated Physics': ['DINO-WM + Estimated Physics', '#8E5B8A', 'h']}


## Read saved results

Read only the files selected above; no model inference or simulation.


In [ ]:
curves = json.loads(curves_file.read_text())
for environment in ('reacher', 'fetch_slide'):
    for method in styles:
        record = curves[environment][method]
        if not record['grid'] or len(record['grid']) != len(record['curve_percent']):
            raise ValueError(f'{curves_file}: invalid grid/curve for {environment}/{method}')
        if record['grid'] != curves[environment]['Vanilla']['grid']:
            raise ValueError(f'{curves_file}: methods must share a grid in {environment}')
inputs = {'curves': str(curves_file.resolve())}
print('Targets per curve:', sorted({r['count'] for values in curves.values() for r in values.values()}))

run_dir = create_output(root / 'logs/paper_figures' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'), {'inputs': inputs})


## Plot and export

Keep the paper layout and export PDF/PNG and plotted values into a fresh output directory.


In [ ]:
dino = ['Vanilla', 'Temporal Straightening', 'Long History', 'Context Prefix',
        'Privileged Physics', 'Context', 'Estimated Physics']
lewm = ['LeWM', 'LeWM + Context']
fig, axes = plt.subplots(2, 2, figsize=(13.5, 11), sharey=False)
for row, methods in enumerate((dino, lewm)):
    family = 'DINO-WM' if row == 0 else 'LeWM'
    for column, environment in enumerate(('reacher', 'fetch_slide')):
        ax = axes[row, column]
        grid = curves[environment]['Vanilla']['grid']
        for method in methods:
            label, color, marker = styles[method]
            ax.plot(grid, curves[environment][method]['curve_percent'],
                    color=color, marker=marker, label=label, linestyle='-', linewidth=2.2,
                    markersize=5, markevery=2, clip_on=False)
        xlabel = 'Joint-angle error tolerance (°)' if column == 0 else 'Error tolerance (cm)'
        ax.set(xlim=(grid[0], grid[-1]), ylim=(0, 100), xlabel=xlabel)
        ax.set_ylabel('Success rate (%)', fontsize=17)
        ax.set_xticks([0, 10, 20, 30] if column == 0 else list(range(0, 11, 2)))
        ax.set_yticks(range(0, 101, 20))
        ax.tick_params(labelsize=16)
        ax.grid(axis='y', color='0.9', linewidth=.7)
        ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)
        panel = 'abcd'[2 * row + column]
        environment_label = 'Reacher' if column == 0 else 'FetchSlide'
        ax.text(.5, -.26, f'({panel}) {environment_label} · {family}-Based Models',
                transform=ax.transAxes, ha='center', va='top', fontsize=19, fontweight='bold')
    handles = [Line2D([], [], color=styles[method][1], marker=styles[method][2],
                      linestyle='-', linewidth=2.2, markersize=5, label=styles[method][0])
               for method in methods]
    fig.legend(handles=handles, loc='upper center', ncol=3 if row == 0 else 2,
               bbox_to_anchor=(.5, .98 if row == 0 else .465), frameon=False, fontsize=14.5,
               handlelength=2, columnspacing=1.8 if row == 0 else 3, handletextpad=.6)
fig.subplots_adjust(left=.08, right=.98, bottom=.08, top=.85, hspace=.60, wspace=.28)
for extension in ('pdf', 'png'):
    fig.savefig(run_dir / f'success_thresholds_split_with_estimated.{extension}', dpi=300, bbox_inches='tight')
(run_dir / 'curves.json').write_text(json.dumps(curves, indent=2) + '\n')
plt.show()
print(run_dir)


Paper styling is retained; plotted values come from the selected inputs. PDF file bytes may differ because of metadata.
